In [1]:
import pandas as pd
import numpy as np
import os
print(os.getcwd())
print(os.listdir())
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib




C:\Users\t490\Smart-healthcare-system
['.git', '.ipynb_checkpoints', '01_Data_Cleaning.ipynb', 'data']


In [2]:
df = pd.read_csv("data/DiseaseAndSymptoms.csv")
print(os.getcwd())

C:\Users\t490\Smart-healthcare-system


In [3]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns)

print(df.head())

Shape: (4920, 18)

Columns:
Index(['Disease', 'Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4',
       'Symptom_5', 'Symptom_6', 'Symptom_7', 'Symptom_8', 'Symptom_9',
       'Symptom_10', 'Symptom_11', 'Symptom_12', 'Symptom_13', 'Symptom_14',
       'Symptom_15', 'Symptom_16', 'Symptom_17'],
      dtype='object')
            Disease   Symptom_1              Symptom_2              Symptom_3  \
0  Fungal infection     itching              skin_rash   nodal_skin_eruptions   
1  Fungal infection   skin_rash   nodal_skin_eruptions    dischromic _patches   
2  Fungal infection     itching   nodal_skin_eruptions    dischromic _patches   
3  Fungal infection     itching              skin_rash    dischromic _patches   
4  Fungal infection     itching              skin_rash   nodal_skin_eruptions   

              Symptom_4 Symptom_5 Symptom_6 Symptom_7 Symptom_8 Symptom_9  \
0   dischromic _patches       NaN       NaN       NaN       NaN       NaN   
1                   NaN       NaN       

In [4]:
df.isnull().sum()

Disease          0
Symptom_1        0
Symptom_2        0
Symptom_3        0
Symptom_4      348
Symptom_5     1206
Symptom_6     1986
Symptom_7     2652
Symptom_8     2976
Symptom_9     3228
Symptom_10    3408
Symptom_11    3726
Symptom_12    4176
Symptom_13    4416
Symptom_14    4614
Symptom_15    4680
Symptom_16    4728
Symptom_17    4848
dtype: int64

In [5]:
df.fillna("none", inplace=True)

In [6]:
df.isnull().sum()

Disease       0
Symptom_1     0
Symptom_2     0
Symptom_3     0
Symptom_4     0
Symptom_5     0
Symptom_6     0
Symptom_7     0
Symptom_8     0
Symptom_9     0
Symptom_10    0
Symptom_11    0
Symptom_12    0
Symptom_13    0
Symptom_14    0
Symptom_15    0
Symptom_16    0
Symptom_17    0
dtype: int64

In [7]:
print("duplicates:",df.duplicated().sum())
print("unique Disease:",df["Disease"].nunique())

duplicates: 4616
unique Disease: 41


In [8]:
print("Before:", df.shape)

df_no_dup = df.drop_duplicates()

print("After:", df_no_dup.shape)

Before: (4920, 18)
After: (304, 18)


In [9]:
symptoms = set()

for col in df.columns[1:]:
    symptoms.update(df[col].astype(str).str.strip().unique())

symptoms.discard("none")

print("Total Symptoms:", len(symptoms))

Total Symptoms: 131


In [10]:
all_symptoms = sorted(list(symptoms))

encoded_df = pd.DataFrame(
    0,
    index=df.index,
    columns=all_symptoms
)

for idx, row in df.iterrows():
    for col in df.columns[1:]:
        symptom = str(row[col]).strip()

        if symptom != "none":
            encoded_df.at[idx, symptom] = 1

encoded_df.head()

,abdominal_pain,abnormal_menstruation,acidity,acute_liver_failure,altered_sensorium,anxiety,back_pain,belly_pain,blackheads,bladder_discomfort,...,vomiting,watering_from_eyes,weakness_in_limbs,weakness_of_one_body_side,weight_gain,weight_loss,yellow_crust_ooze,yellow_urine,yellowing_of_eyes,yellowish_skin
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [11]:
encoded_df.sum().sort_values(ascending=False).head(20)

fatigue              1932
vomiting             1914
high_fever           1362
loss_of_appetite     1152
nausea               1146
headache             1134
abdominal_pain       1032
yellowish_skin        912
yellowing_of_eyes     816
chills                798
skin_rash             786
malaise               702
chest_pain            696
joint_pain            684
itching               678
sweating              678
dark_urine            570
diarrhoea             564
cough                 564
muscle_pain           474
dtype: int64

In [12]:
print(encoded_df.shape)

(4920, 131)


In [13]:
print(encoded_df.shape)

(4920, 131)


In [14]:
print(encoded_df.shape)

print(encoded_df.iloc[0].sum())

encoded_df.sum().sort_values(ascending=False).head(10)

(4920, 131)
4


fatigue              1932
vomiting             1914
high_fever           1362
loss_of_appetite     1152
nausea               1146
headache             1134
abdominal_pain       1032
yellowish_skin        912
yellowing_of_eyes     816
chills                798
dtype: int64

In [15]:
le = LabelEncoder()
y = le.fit_transform(df["Disease"])

print("Total Diseases:", len(le.classes_))

Total Diseases: 41


In [16]:


X_train, X_test, y_train, y_test = train_test_split(
    encoded_df,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(3936, 131)
(984, 131)


In [17]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [18]:
y_pred = rf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 1.0


In [19]:
os.makedirs("models", exist_ok=True)

joblib.dump(rf, "models/disease_model.pkl")
joblib.dump(le, "models/label_encoder.pkl")

['models/label_encoder.pkl']

In [20]:
print(os.listdir("models"))

['disease_model.pkl', 'label_encoder.pkl']
